# COMP 4330SEF P04 Group 5: LightRAG vs Naive RAG (Experiment 1)

This notebook runs the whole pipeline: **setup → data → naive RAG baseline → LightRAG index → answering → evaluation**.

**Before running**
1. Choose a backend in the config cell:
   * `BACKEND = "gemini"`: Gemini API (`gemini-2.5-flash-lite` plus `gemini-embedding-001`). A **CPU runtime** is enough. Add your key under Colab's 🔑 **Secrets** panel with the name `GEMINI_API_KEY` (the name `OPENAI_API_KEY` also works) and turn on notebook access.
   * `BACKEND = "ollama"`: free local `qwen2.5:14b` plus `bge-m3`. It needs a **T4 GPU runtime** and is much slower (about 3–6 hours to index).
2. Run all cells with **Runtime → Run all**.

Progress is saved to Google Drive (`MyDrive/COMP4330_LightRAG`). If Colab disconnects, run all again. Indexing, answering and scoring continue from where they stopped.

In [ ]:
%pip install -q lightrag-hku ragas "langchain-community<0.4" openai tiktoken huggingface_hub pandas

## 1. Config

In [ ]:
import os, sys, json, time, math, re, string, hashlib, collections, subprocess
from pathlib import Path
import numpy as np

IN_COLAB = "google.colab" in sys.modules
BACKEND = "gemini"            # "gemini" (Google API key) or "ollama" (free local backup, needs T4 GPU)

TARGET_TOKENS = 100_000       # corpus size: shortest UltraDomain-mix documents until this many tokens
QUESTION_LIMIT = None         # e.g. 5 for a quick trial run; None = all questions
CHUNK_SIZE, CHUNK_OVERLAP = 1200, 100   # LightRAG defaults, used by both systems
NAIVE_TOP_K = 8               # 8 x 1200 tokens ~= 9.6k, inside the shared context budget
MAX_CONTEXT_TOKENS = 12_000   # retrieved-context budget for LightRAG

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/COMP4330_LightRAG")
else:
    ROOT = Path("colab_local_test")
DATA = ROOT / "data"
RESULTS = ROOT / f"results_{BACKEND}"   # separate per backend: embedding sizes differ
for p in (DATA / "corpus", RESULTS):
    p.mkdir(parents=True, exist_ok=True)

if BACKEND == "gemini":
    # Gemini's OpenAI-compatible endpoint, so every call below works unchanged with a different base_url
    if IN_COLAB:
        from google.colab import userdata
        for _name in ("GEMINI_API_KEY", "OPENAI_API_KEY"):   # the Secret may carry either name; the value is the Gemini key
            try:
                os.environ["GEMINI_API_KEY"] = userdata.get(_name)
                break
            except Exception:
                pass
    assert os.environ.get("GEMINI_API_KEY"), "Add your Gemini key as a Colab Secret (GEMINI_API_KEY) and turn on Notebook access"
    BASE_URL, API_KEY = "https://generativelanguage.googleapis.com/v1beta/openai/", os.environ["GEMINI_API_KEY"]
    # flash-lite does not "think" by default, so completion tokens stay comparable with other hosted models.
    # If you switch to gemini-2.5-flash, set LLM_EXTRA = {"reasoning_effort": "none"}.
    LLM_MODEL, EMBED_MODEL, EMBED_DIM = "gemini-2.5-flash-lite", "gemini-embedding-001", 3072
    LLM_EXTRA, MAX_ASYNC = {}, 3
else:
    BASE_URL, API_KEY = "http://localhost:11434/v1", "ollama"
    LLM_MODEL, EMBED_MODEL, EMBED_DIM = "qwen2.5-14b-16k", "bge-m3", 1024
    # qwen2.5:14b over-extracts entities and can loop until the context is full; cap the output.
    LLM_EXTRA, MAX_ASYNC = {"max_tokens": 3000}, 2
print("backend:", BACKEND, LLM_MODEL, EMBED_MODEL, "| saving to", ROOT)

## 2. Local model server (only for `BACKEND = "ollama"`)

In [ ]:
if BACKEND == "ollama":
    if IN_COLAB:
        subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
        subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(5)
    for m in ["qwen2.5:14b", "bge-m3"]:
        subprocess.run(["ollama", "pull", m], check=True, capture_output=True)
    # Ollama's OpenAI-compatible API cannot set the context size per request, so bake 16k into a model variant.
    Path("Modelfile").write_text("FROM qwen2.5:14b\nPARAMETER num_ctx 16384\nPARAMETER temperature 0\n")
    subprocess.run(["ollama", "create", LLM_MODEL, "-f", "Modelfile"], check=True, capture_output=True)
    print("ollama ready")

## 3. Data: UltraDomain `mix` subset
We keep the shortest documents until the total reaches `TARGET_TOKENS`, together with every question about them. Each question records its source document (`doc_id`), which serves as the ground truth for hit rate and nDCG.

In [ ]:
import tiktoken
from huggingface_hub import hf_hub_download

ENC = tiktoken.get_encoding("cl100k_base")
rows = [json.loads(l) for l in open(hf_hub_download("TommyChien/UltraDomain", "mix.jsonl", repo_type="dataset"), encoding="utf-8")]
doc_ids = {}
for r in rows:
    doc_ids.setdefault(r["context"], "doc_" + hashlib.md5(r["context"].encode()).hexdigest()[:10])

picked, total = set(), 0
for text in sorted(doc_ids, key=lambda t: len(ENC.encode(t))):
    n = len(ENC.encode(text))
    if total + n > TARGET_TOKENS:
        break
    picked.add(text); total += n

for text in picked:
    (DATA / "corpus" / f"{doc_ids[text]}.txt").write_text(text, encoding="utf-8")
QUESTIONS = [{"qid": f"q{i:03d}", "question": r["input"], "answers": r["answers"],
              "doc_id": doc_ids[r["context"]], "dataset": r["dataset"]}
             for i, r in enumerate(rows) if r["context"] in picked][:QUESTION_LIMIT]
DOCS = sorted((DATA / "corpus").glob("*.txt"))
DOCS = [p for p in DOCS if p.read_text(encoding="utf-8") in picked]
print(f"{len(DOCS)} documents, {total} tokens, {len(QUESTIONS)} questions")
print(collections.Counter(q["dataset"] for q in QUESTIONS))

## 4. Naive RAG baseline
Fixed-size chunks → embeddings → cosine top-k → one LLM call.

In [ ]:
from openai import OpenAI, AsyncOpenAI
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
aclient = AsyncOpenAI(base_url=BASE_URL, api_key=API_KEY)

import asyncio

def _rate_limited(e):
    return "429" in str(e) or "RESOURCE_EXHAUSTED" in str(e)

def _wait(e):                     # Gemini says "Please retry in 51.9s"
    m = re.search(r"retry in ([\d.]+)s", str(e))
    return float(m.group(1)) + 1 if m else 30

def call_retry(fn, tries=10):     # per-minute quotas (Gemini free tier) are waited out instead of crashing the run
    for n in range(tries):
        try:
            return fn()
        except Exception as e:
            if not _rate_limited(e) or n == tries - 1:
                raise
            print(f"  rate limited, waiting {_wait(e):.0f}s"); time.sleep(_wait(e))

async def acall_retry(fn, tries=10):
    for n in range(tries):
        try:
            return await fn()
        except Exception as e:
            if not _rate_limited(e) or n == tries - 1:
                raise
            print(f"  rate limited, waiting {_wait(e):.0f}s"); await asyncio.sleep(_wait(e))

# Gemini free tier allows 30k embedding tokens per minute: one batch of 32 x 1200 tokens would exceed it on its own
EMBED_BATCH = 16 if BACKEND == "gemini" else 32

NAIVE_PROMPT = """Answer the question using only the context below. If the context does not contain the answer, say so.
Reply in a single paragraph.

---Context---
{context}

---Question---
{question}"""

def chunk(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    toks = ENC.encode(text)
    return [ENC.decode(toks[i:i + size]) for i in range(0, max(len(toks) - overlap, 1), size - overlap)]

def embed_sync(texts):
    vecs = np.array([d.embedding for d in call_retry(lambda: client.embeddings.create(model=EMBED_MODEL, input=texts)).data], dtype=np.float32)
    return vecs / np.linalg.norm(vecs, axis=1, keepdims=True)

def top_k(query_vec, matrix, k):
    return np.argsort(-(matrix @ query_vec))[:k]

NAIVE_DIR = RESULTS / "naive_index"
if (NAIVE_DIR / "vectors.npy").exists():
    CHUNKS = json.loads((NAIVE_DIR / "chunks.json").read_text(encoding="utf-8"))
    VECTORS = np.load(NAIVE_DIR / "vectors.npy")
else:
    CHUNKS = [{"doc_id": p.stem, "text": c} for p in DOCS for c in chunk(p.read_text(encoding="utf-8"))]
    VECTORS = np.concatenate([embed_sync([c["text"] for c in CHUNKS[i:i + EMBED_BATCH]]) for i in range(0, len(CHUNKS), EMBED_BATCH)])
    NAIVE_DIR.mkdir(exist_ok=True)
    np.save(NAIVE_DIR / "vectors.npy", VECTORS)
    (NAIVE_DIR / "chunks.json").write_text(json.dumps(CHUNKS, ensure_ascii=False), encoding="utf-8")

def naive_answer(question):
    t0 = time.perf_counter()
    retrieved = [CHUNKS[i] for i in top_k(embed_sync([question])[0], VECTORS, NAIVE_TOP_K)]
    context = "\n\n".join(f"[{n + 1}] {c['text']}" for n, c in enumerate(retrieved))
    r = call_retry(lambda: client.chat.completions.create(model=LLM_MODEL, temperature=0, **LLM_EXTRA,
        messages=[{"role": "user", "content": NAIVE_PROMPT.format(context=context, question=question)}]))
    return {"answer": r.choices[0].message.content,
            "retrieved_doc_ids": [c["doc_id"] for c in retrieved], "contexts": [c["text"] for c in retrieved],
            "prompt_tokens": r.usage.prompt_tokens, "completion_tokens": r.usage.completion_tokens,
            "llm_calls": 1, "latency_s": round(time.perf_counter() - t0, 2)}

# self-check
assert list(top_k(np.array([0.1, 0.9, 0.2]), np.eye(3), 2)) == [1, 2]
assert len(chunk("word " * 3000)) == 3
print(len(CHUNKS), "chunks indexed for naive RAG")

## 5. LightRAG index
The LLM extracts entities and relations from every chunk to build the knowledge graph. This is the slow and costly step. It resumes automatically after a disconnect.

In [ ]:
from lightrag import LightRAG, QueryParam
from lightrag.llm.openai import openai_complete_if_cache
from lightrag.utils import EmbeddingFunc, TokenTracker

TRACKER = TokenTracker()

async def llm(prompt, system_prompt=None, history_messages=None, **kw):
    return await acall_retry(lambda: openai_complete_if_cache(LLM_MODEL, prompt, system_prompt=system_prompt,
        history_messages=history_messages or [], base_url=BASE_URL, api_key=API_KEY,
        token_tracker=TRACKER, **{"temperature": 0, **LLM_EXTRA, **kw}))

async def embed(texts, **_):
    r = await acall_retry(lambda: aclient.embeddings.create(model=EMBED_MODEL, input=texts))
    return np.array([d.embedding for d in r.data], dtype=np.float32)

rag = LightRAG(
    working_dir=str(RESULTS / "lightrag_index"),
    llm_model_func=llm, llm_model_name=LLM_MODEL, llm_model_max_async=MAX_ASYNC,
    embedding_func=EmbeddingFunc(embedding_dim=EMBED_DIM, max_token_size=8192, func=embed),
    chunk_token_size=CHUNK_SIZE, chunk_overlap_token_size=CHUNK_OVERLAP,
    enable_llm_cache=False,                     # measure query cost fresh every time
    enable_llm_cache_for_entity_extract=True,   # makes an interrupted index cheap to resume
)
await rag.initialize_storages()

TRACKER.reset(); t0 = time.perf_counter()
await rag.ainsert([p.read_text(encoding="utf-8") for p in DOCS], ids=[p.stem for p in DOCS],
                  file_paths=[p.name for p in DOCS])
this_run = {"index_time_s": round(time.perf_counter() - t0, 1), **TRACKER.get_usage()}
_p = RESULTS / "index_cost.json"
index_cost = json.loads(_p.read_text()) if _p.exists() else {}
for k, v in this_run.items():          # add to what earlier runs recorded; a rerun after a finished index adds ~0
    index_cost[k] = round(index_cost.get(k, 0) + v, 1)
index_cost["docs"] = len(DOCS)
_p.write_text(json.dumps(index_cost, indent=2))
print("this run:", this_run)
print("cumulative LightRAG index cost (lower bound if a run was interrupted):", index_cost)


## 6. Answer every question with both systems
LightRAG uses its default `mix` mode with reranking off. The results are written to `exp1_naive.jsonl` and `exp1_lightrag.jsonl`.

In [ ]:
async def lightrag_answer(question, mode="mix"):
    TRACKER.reset(); t0 = time.perf_counter()
    r = await rag.aquery_llm(question, param=QueryParam(mode=mode, response_type="Single Paragraph",
                                                        enable_rerank=False, max_total_tokens=MAX_CONTEXT_TOKENS))
    data, usage = r.get("data") or {}, TRACKER.get_usage()
    chunks = data.get("chunks", [])
    # Faithfulness is judged against everything the LLM saw: chunk text AND entity/relation descriptions.
    kg_text = [f"{e.get('entity_name')}: {e.get('description')}" for e in data.get("entities", [])] + \
              [f"{r.get('src_id')} -> {r.get('tgt_id')}: {r.get('description')}" for r in data.get("relationships", [])]
    return {"answer": (r.get("llm_response") or {}).get("content", ""),
            "retrieved_doc_ids": [c.get("file_path", "").removesuffix(".txt") for c in chunks],
            "contexts": [c.get("content", "") for c in chunks] + kg_text,
            "n_entities": len(data.get("entities", [])), "n_relations": len(data.get("relationships", [])),
            "prompt_tokens": usage["prompt_tokens"], "completion_tokens": usage["completion_tokens"],
            "llm_calls": usage["call_count"], "latency_s": round(time.perf_counter() - t0, 2)}

def load_jsonl(path):
    return [json.loads(l) for l in open(path, encoding="utf-8")] if path.exists() else []

async def naive_async(q):
    return naive_answer(q)

for name, answer_fn in [("naive", naive_async), ("lightrag", lightrag_answer)]:
    out = RESULTS / f"exp1_{name}.jsonl"
    done = {r["qid"] for r in load_jsonl(out)}
    for i, q in enumerate(QUESTIONS):
        if q["qid"] in done:
            continue
        res = await answer_fn(q["question"])
        with out.open("a", encoding="utf-8") as f:
            f.write(json.dumps({**q, "system": name, **res}, ensure_ascii=False) + "\n")
        print(f"[{name}] {i + 1}/{len(QUESTIONS)} {q['qid']} {res['latency_s']}s")
print("done")

## 7. Evaluation
* **Retrieval: hit rate and nDCG.** These are measured at document level. A retrieved chunk counts as relevant when it comes from the question's source document.
* **Generation: RAGAS faithfulness and answer relevancy** (Es et al., 2024). The same LLM acts as the judge. This is a known bias, so it should be listed as a limitation.
* **Token F1** against the gold answer. This is a cheap sanity check.
* **Cost:** tokens, LLM calls and latency, copied from the run.

In [ ]:
from ragas.llms import llm_factory
from ragas.embeddings import embedding_factory
from ragas.metrics.collections import Faithfulness, AnswerRelevancy

def strip_references(answer):
    return re.split(r"\n#*\s*References\s*\n", answer)[0].strip()   # LightRAG appends a reference list

def norm_tokens(text):
    text = "".join(ch for ch in text.lower() if ch not in string.punctuation)
    return [w for w in text.split() if w not in {"a", "an", "the"}]

def token_f1(pred, golds):
    best = 0.0
    for gold in golds:
        common = sum((collections.Counter(norm_tokens(pred)) & collections.Counter(norm_tokens(gold))).values())
        if common:
            p, r = common / len(norm_tokens(pred)), common / len(norm_tokens(gold))
            best = max(best, 2 * p * r / (p + r))
    return best

def ndcg(retrieved, gold, n_gold_chunks):
    dcg = sum(1 / math.log2(i + 2) for i, d in enumerate(retrieved) if d == gold)
    ideal = sum(1 / math.log2(i + 2) for i in range(min(len(retrieved), n_gold_chunks)))
    return dcg / ideal if ideal else 0.0

assert strip_references("Ans.\n\n### References\n\n- [1] a.txt") == "Ans."
assert token_f1("The cat sat", ["cat sat down"]) == 0.8
assert ndcg(["a", "b"], "a", 1) == 1.0 and ndcg(["b", "a"], "a", 1) < 1 and ndcg(["b"], "a", 3) == 0

judge = llm_factory(LLM_MODEL, client=aclient, temperature=0, max_tokens=3000)
faith = Faithfulness(llm=judge)
relevancy = AnswerRelevancy(llm=judge, embeddings=embedding_factory("openai", EMBED_MODEL, client=aclient))
n_gold_chunks = collections.Counter(c["doc_id"] for c in CHUNKS)

SCORES = RESULTS / "scores.jsonl"

async def score_files(paths, scores_path):
    scored = {(s["system"], s["qid"]) for s in load_jsonl(scores_path)}
    for path in paths:
        records = load_jsonl(path)
        for i, rec in enumerate(records):
            name = rec["system"]
            if (name, rec["qid"]) in scored:
                continue
            answer = strip_references(rec["answer"])
            s = {"system": name, "qid": rec["qid"],
                 "hit": float(rec["doc_id"] in rec["retrieved_doc_ids"]),
                 "ndcg": ndcg(rec["retrieved_doc_ids"], rec["doc_id"], n_gold_chunks[rec["doc_id"]]),
                 "token_f1": token_f1(answer, rec["answers"]),
                 "n_entities": rec.get("n_entities"), "n_relations": rec.get("n_relations"),
                 **{k: rec[k] for k in ("prompt_tokens", "completion_tokens", "llm_calls", "latency_s")}}
            for metric, make in [("faithfulness", lambda: faith.ascore(user_input=rec["question"], response=answer,
                                                                     retrieved_contexts=rec["contexts"] or [""])),
                                 ("answer_relevancy", lambda: relevancy.ascore(user_input=rec["question"], response=answer))]:
                try:
                    s[metric] = (await acall_retry(make)).value
                except Exception as e:   # one malformed judge reply loses one score, not the run
                    print(f"  {metric} failed for {name}/{rec['qid']}: {str(e)[:120]}")
                    s[metric] = None
            with scores_path.open("a", encoding="utf-8") as f:
                f.write(json.dumps(s) + "\n")
            print(f"[{name}] {i + 1}/{len(records)} faith={s['faithfulness']} rel={s['answer_relevancy']} hit={s['hit']}")
    print("scoring done")

await score_files([RESULTS / "exp1_naive.jsonl", RESULTS / "exp1_lightrag.jsonl"], SCORES)

## 8. Results table (copy into slides and the report)

In [ ]:
import pandas as pd
df = pd.DataFrame(load_jsonl(SCORES))
summary = df.groupby("system")[["hit", "ndcg", "faithfulness", "answer_relevancy", "token_f1",
                                "prompt_tokens", "completion_tokens", "llm_calls", "latency_s"]].mean().round(3)
summary.insert(0, "n", df.groupby("system").size())
summary.to_csv(RESULTS / "exp1_summary.csv")
print("LightRAG index cost:", json.loads((RESULTS / "index_cost.json").read_text()))
summary

## 9. Experiment 3: LightRAG query modes
Same index, same questions, same LLM. Only `QueryParam.mode` changes: `naive` (chunk vector search only, through LightRAG), `local` (entity-level), `global` (relation-level), `hybrid` (local + global) and `mix` (hybrid + chunk vector search). `mix` is exactly the Exp 1 LightRAG run, so its answers are reused instead of being paid for twice.

**Check the keyword diagnostic first.** In `local`, `global` and `hybrid`, LightRAG decides what to retrieve from two keyword lists that the LLM extracts from each question: *low-level* (entities) and *high-level* (themes). If the LLM returns an empty list, the matching branch silently does nothing, and in LightRAG 1.5 a `local` query with no low-level keywords falls through to the hybrid code path. In that case several modes return identical context, and differences between them mean nothing. A local run with `qwen2.5:14b` showed this: on the 46 questions the low-level list was empty for 16 (35%) and the high-level list for none. So the result of the next cell decides how the mode comparison should be read.

In [ ]:
from lightrag.operate import get_keywords_from_query

_gc = rag._build_global_config()
_kw = [await get_keywords_from_query(q["question"], QueryParam(mode="mix"), _gc, rag.llm_response_cache) for q in QUESTIONS]
KEYWORD_DIAG = {"questions": len(_kw), "empty_low_level": sum(not ll for hl, ll in _kw),
                "empty_high_level": sum(not hl for hl, ll in _kw)}
(RESULTS / "exp3_keyword_diagnostic.json").write_text(json.dumps(KEYWORD_DIAG, indent=2))
print(KEYWORD_DIAG)

In [ ]:
MODES = ["naive", "local", "global", "hybrid", "mix"]
qids = {q["qid"] for q in QUESTIONS}

def append(path, rec):
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

for mode in MODES:
    out = RESULTS / f"exp3_{mode}.jsonl"
    done = {r["qid"] for r in load_jsonl(out)}
    if mode == "mix":   # same call as Exp 1's LightRAG run
        for r in load_jsonl(RESULTS / "exp1_lightrag.jsonl"):
            if r["qid"] in qids and r["qid"] not in done:
                append(out, {**r, "system": "lightrag-mix"})
        continue
    for i, q in enumerate(QUESTIONS):
        if q["qid"] in done:
            continue
        res = await lightrag_answer(q["question"], mode=mode)
        append(out, {**q, "system": f"lightrag-{mode}", **res})
        print(f"[{mode}] {i + 1}/{len(QUESTIONS)} {q['qid']} {res['latency_s']}s")
print("done")

In [ ]:
E3_SCORES = RESULTS / "exp3_scores.jsonl"
await score_files([RESULTS / f"exp3_{m}.jsonl" for m in MODES], E3_SCORES)

df3 = pd.DataFrame(load_jsonl(E3_SCORES))
summary3 = df3.groupby("system")[["hit", "ndcg", "faithfulness", "answer_relevancy", "token_f1", "n_entities",
                                  "n_relations", "prompt_tokens", "completion_tokens", "llm_calls", "latency_s"]].mean().round(3)
summary3.insert(0, "n", df3.groupby("system").size())
summary3 = summary3.reindex([f"lightrag-{m}" for m in MODES])
summary3.to_csv(RESULTS / "exp3_summary.csv")
print("keyword diagnostic:", KEYWORD_DIAG)
summary3

**How to read the table.** If the diagnostic shows that most questions have empty low-level (or high-level) keywords, `local`, `global` and `hybrid` are not three different retrievers in this setup. Check `n_entities` and `n_relations`: identical values across modes mean identical retrieved context. Report that finding instead of ranking the modes. With a stronger hosted LLM (such as Gemini) the keyword lists are normally non-empty, and the modes should then differ.

## 10. Export for the report
Writes two files next to the results in Google Drive: `per_question.csv` (one row per question and system, with answers and every score) and `run_info.json` (models, settings, corpus size, index cost). Use these files, not screenshots, for the report and slides. **Only run this after the full run** (`QUESTION_LIMIT = None`): the files cover whatever has been answered and scored so far.

`mode` is `naive_rag` for the baseline and `lightrag-<mode>` for LightRAG. There is no correctness-judge column: the closest are `token_f1` and the two RAGAS scores.

In [ ]:
import csv

E3_SCORES = RESULTS / "exp3_scores.jsonl"
q_by_id = {q["qid"]: q for q in QUESTIONS}
# Exp 1 gives naive_rag and lightrag-mix; Exp 3 adds the other LightRAG modes (its mix is the Exp 1 run, so skip it)
sources = [("exp1", "naive", "naive_rag", SCORES), ("exp1", "lightrag", "lightrag-mix", SCORES)] + \
          [("exp3", m, f"lightrag-{m}", E3_SCORES) for m in ["naive", "local", "global", "hybrid"]]
rows = []
for exp, name, mode, score_file in sources:
    system = name if exp == "exp1" else f"lightrag-{name}"
    scores = {s["qid"]: s for s in load_jsonl(score_file) if s["system"] == system}
    for r in load_jsonl(RESULTS / f"{exp}_{name}.jsonl"):
        s = scores.get(r["qid"])
        if s is None:
            continue
        rows.append({"experiment": exp, "qid": r["qid"], "question": r["question"], "system": system, "mode": mode,
                     "hit": s["hit"], "ndcg": round(s["ndcg"], 4), "answer": r["answer"],
                     "gold_answer": " | ".join(r["answers"]), "token_f1": round(s["token_f1"], 4),
                     "faithfulness": s["faithfulness"], "answer_relevancy": s["answer_relevancy"],
                     "n_entities": s.get("n_entities"), "n_relations": s.get("n_relations"),
                     "prompt_tokens": s["prompt_tokens"], "completion_tokens": s["completion_tokens"],
                     "llm_calls": s["llm_calls"], "latency_s": s["latency_s"],
                     "dataset": q_by_id[r["qid"]]["dataset"], "source_doc": r["doc_id"]})
with open(RESULTS / "per_question.csv", "w", newline="", encoding="utf-8-sig") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0]))
    w.writeheader(); w.writerows(rows)

def _read(p):
    return json.loads(p.read_text()) if p.exists() else None

texts = [p.read_text(encoding="utf-8") for p in DOCS]
run_info = {
    "backend": BACKEND, "llm": LLM_MODEL, "embedding": EMBED_MODEL, "embedding_dim": EMBED_DIM,
    "judge": LLM_MODEL + " (same model, RAGAS)", "chunk_size": CHUNK_SIZE, "chunk_overlap": CHUNK_OVERLAP,
    "naive_top_k": NAIVE_TOP_K, "lightrag_top_k": rag.top_k, "lightrag_chunk_top_k": rag.chunk_top_k,
    "lightrag_max_context_tokens": MAX_CONTEXT_TOKENS, "reranker": False,
    "documents": len(DOCS), "corpus_tokens": sum(len(ENC.encode(t)) for t in texts),
    "corpus_words": sum(len(t.split()) for t in texts), "chunks": len(CHUNKS), "questions": len(QUESTIONS),
    "question_limit": QUESTION_LIMIT, "index_cost": _read(RESULTS / "index_cost.json"),
    "keyword_diagnostic": _read(RESULTS / "exp3_keyword_diagnostic.json"),
}
(RESULTS / "run_info.json").write_text(json.dumps(run_info, indent=2))
print(len(rows), "rows ->", RESULTS / "per_question.csv")
print(json.dumps(run_info, indent=2))

---
## Appendix: results already obtained locally (Ollama, qwen2.5:14b + bge-m3)

These numbers were produced **before** the Colab run, on a local RTX 5060 Ti with **`BACKEND = "ollama"`**. The LLM is `qwen2.5:14b`, not a hosted model, so they will not match a Colab run with the `gemini` backend. Corpus: 20 UltraDomain-`mix` documents (95k tokens), 46 questions. The same LLM, embeddings, 1200-token chunks and ~12k context budget were used for both systems.

| System | n | Hit rate | nDCG | Faithfulness | Answer relevancy | Token F1 | Prompt tokens / query | Completion tokens / query | LLM calls / query | Latency (s) |
|---|---|---|---|---|---|---|---|---|---|---|
| Naive RAG | 46 | 0.957 | 0.821 | **0.899** | 0.834 | **0.403** | **9,267** | **142** | 1 | **10.9** |
| LightRAG (mix) | 46 | 0.913 | **0.833** | 0.829 | **0.859** | 0.358 | 12,206 | 243 | 2 | 18.1 |

**LightRAG index cost (one-off):** 20 documents, 156 LLM calls, 681,579 prompt + 334,531 completion tokens, **161 minutes**. Naive RAG only embeds the chunks.

**What the numbers say**
* **LightRAG did not beat naive RAG here.** Naive RAG is ahead on token F1 (wins on 33 of 46 questions, LightRAG on 13), faithfulness (naive ahead on 14, LightRAG on 9, 23 ties), hit rate, and cost per query. LightRAG leads slightly on answer relevancy (0.859 vs 0.834) and nDCG (0.833 vs 0.821). The differences in relevancy and nDCG are small, and with n = 46 and a single run we do not claim they are significant.
* **The advantage reported in the LightRAG paper is not reproduced in this setup.** One likely reason is that the corpus is small (95k tokens). The questions are also mostly single-document: only 19 of 46 (2WikiMQA 16, HotpotQA 3) are multi-hop, while qasper has 19 and multifieldqa 8. A graph index has the most to offer on large corpora and global questions.
* **Entity extraction was truncated.** `qwen2.5:14b` over-extracts and loops, so its output was capped at 3,000 tokens. Many chunks hit that cap (the log warns about truncation in most documents), so the knowledge graph lacks some entities and relations. This probably hurts LightRAG. It is also a concrete motivation for the planned improvement (tuning the extraction prompt).
* **Low-level keywords were often empty.** For 16 of the 46 questions `qwen2.5:14b` returned no low-level (entity) keywords when LightRAG asked it to extract them, while high-level keywords were always present. For those questions the entity-level branch of retrieval did nothing, so LightRAG's `mix` run there was relation-level retrieval plus chunk vector search. A stronger LLM may not show this.
* **Faithfulness for LightRAG was scored against chunk text only** in this local run. The entity and relation descriptions that LightRAG also gave to the LLM were not part of the judge's context, so its faithfulness (0.829) is probably underestimated. The notebook above now includes them, so a rerun is not directly comparable on that one metric.
* **Caveats:** the judge model is the same model that wrote the answers, hit rate and nDCG are measured at document level (a chunk is relevant if it comes from the question's source document), and rerankers were turned off.

In [ ]:
# Per-question scores from the local run (embedded so they are not lost). Nothing here needs a GPU or API.
import io, pandas as pd
LOCAL_CSV = '''system,qid,hit,ndcg,faithfulness,answer_relevancy,token_f1,prompt_tokens,completion_tokens,llm_calls,latency_s
naive,q007,1.0,1.0,1.0,0.9987,0.2778,9168,152,1,13.31
naive,q008,1.0,1.0,1.0,0.8331,0.5714,8692,92,1,9.23
naive,q009,1.0,1.0,1.0,0.9959,0.4762,9415,80,1,9.38
naive,q017,1.0,1.0,0.5714,0.9622,0.3359,8043,130,1,9.86
naive,q018,1.0,1.0,1.0,0.9991,0.3624,9113,138,1,10.02
naive,q019,1.0,1.0,1.0,0.9229,0.3676,8995,120,1,10.24
naive,q020,1.0,1.0,1.0,0.9496,0.4262,9617,65,1,9.14
naive,q021,1.0,1.0,1.0,1.0,0.4324,9496,120,1,10.7
naive,q022,1.0,0.7751,1.0,0.8537,0.4536,9969,93,1,10.39
naive,q023,1.0,1.0,1.0,0.7957,0.3673,9398,97,1,8.53
naive,q024,1.0,0.9931,1.0,0.9809,0.4425,9301,103,1,10.05
naive,q025,1.0,0.4935,0.5,0.0,0.4348,9153,55,1,8.49
naive,q028,1.0,0.4539,1.0,0.9033,0.8282,9746,176,1,12.61
naive,q029,1.0,0.4197,1.0,0.9433,0.4649,9303,215,1,13.36
naive,q030,1.0,1.0,0.9167,1.0,0.4183,8080,153,1,10.46
naive,q031,1.0,1.0,1.0,0.9886,0.2933,8786,160,1,10.5
naive,q032,1.0,1.0,0.8333,0.8755,0.4697,8817,121,1,8.54
naive,q033,1.0,1.0,1.0,0.8712,0.4472,8081,174,1,8.87
naive,q039,1.0,0.7679,1.0,0.9707,0.4578,9590,115,1,10.54
naive,q040,1.0,0.9747,0.3333,0.0,0.3582,9839,72,1,9.62
naive,q045,1.0,1.0,0.4286,0.8792,0.3151,8311,165,1,11.03
naive,q046,1.0,1.0,0.75,0.0,0.4158,8992,104,1,9.77
naive,q050,1.0,1.0,1.0,0.9426,0.2708,9397,358,1,17.74
naive,q051,1.0,1.0,1.0,0.9601,0.7129,9168,72,1,8.99
naive,q052,1.0,0.7877,1.0,0.9541,0.3478,8988,137,1,10.71
naive,q058,1.0,0.8319,1.0,0.9784,0.5057,9506,166,1,12.03
naive,q059,1.0,0.8319,0.2857,0.6042,0.5,9142,102,1,7.63
naive,q068,1.0,0.4693,1.0,0.8778,0.4512,9313,181,1,12.35
naive,q069,1.0,1.0,0.7,0.8822,0.3624,9771,157,1,11.38
naive,q070,1.0,1.0,0.5833,0.9114,0.2129,9099,294,1,14.81
naive,q071,1.0,0.6448,1.0,0.8869,0.3977,10311,162,1,12.75
naive,q072,1.0,0.7115,1.0,0.9413,0.4535,10160,227,1,14.55
naive,q073,1.0,0.6013,0.9375,0.9466,0.3699,9472,204,1,13.23
naive,q076,1.0,0.1303,1.0,0.6218,0.1772,9327,76,1,9.26
naive,q077,1.0,0.7457,1.0,0.8967,0.2889,8175,179,1,11.19
naive,q078,1.0,0.6238,1.0,1.0,0.5263,8817,61,1,8.26
naive,q086,1.0,0.8829,1.0,0.9861,0.2979,9759,197,1,13.2
naive,q087,1.0,1.0,1.0,0.8801,0.4,9738,163,1,12.23
naive,q088,1.0,0.8829,1.0,0.9732,0.3646,9756,175,1,7.11
naive,q090,1.0,1.0,0.7778,0.8807,0.4364,10048,161,1,12.42
naive,q091,1.0,1.0,1.0,0.9885,0.5031,9212,151,1,11.44
naive,q108,0.0,0.0,1.0,0.9109,0.1509,10008,113,1,11.0
naive,q109,0.0,0.0,1.0,0.7479,0.2735,9296,97,1,10.06
naive,q110,1.0,0.9896,1.0,0.8782,0.2111,9484,200,1,13.39
naive,q114,1.0,0.7654,0.8333,0.9705,0.5179,9139,91,1,9.78
naive,q115,1.0,1.0,0.9,0.0,0.3709,9293,118,1,10.73
lightrag,q007,1.0,1.0,1.0,0.9987,0.2088,11370,253,2,17.97
lightrag,q008,1.0,1.0,1.0,0.9117,0.3094,12403,256,2,18.55
lightrag,q009,1.0,1.0,1.0,0.8949,0.2449,12612,276,2,19.76
lightrag,q017,1.0,1.0,0.2727,0.9483,0.2805,11573,217,2,16.47
lightrag,q018,1.0,1.0,1.0,0.9991,0.3034,12562,243,2,18.33
lightrag,q019,1.0,1.0,1.0,0.9412,0.3423,11676,286,2,18.72
lightrag,q020,1.0,1.0,1.0,0.9496,0.3061,12528,180,2,16.35
lightrag,q021,1.0,1.0,1.0,0.9362,0.2526,12655,314,2,20.94
lightrag,q022,1.0,1.0,0.6667,0.8424,0.2774,12659,231,2,20.0
lightrag,q023,1.0,1.0,1.0,0.9368,0.1923,12812,161,2,17.64
lightrag,q024,1.0,1.0,1.0,0.9711,0.3964,11839,150,2,14.57
lightrag,q025,1.0,0.7654,1.0,1.0,0.6065,11792,215,2,16.4
lightrag,q028,0.0,0.0,0.0,0.9995,0.8257,12607,311,2,22.41
lightrag,q029,1.0,0.6367,1.0,0.789,0.3953,12572,390,2,22.78
lightrag,q030,1.0,1.0,0.8889,0.8813,0.3718,11720,229,2,16.78
lightrag,q031,1.0,1.0,1.0,0.9756,0.2057,11905,273,2,18.31
lightrag,q032,1.0,1.0,0.9091,0.9289,0.3043,12225,277,2,18.74
lightrag,q033,1.0,1.0,1.0,0.853,0.3567,11507,223,2,16.31
lightrag,q039,1.0,0.6131,1.0,0.9688,0.4368,12290,166,2,15.48
lightrag,q040,1.0,1.0,0.3333,0.0,0.2245,12899,153,2,17.6
lightrag,q045,1.0,1.0,0.0,0.5595,0.2734,11752,226,2,16.74
lightrag,q046,1.0,1.0,0.8,0.5445,0.5432,12862,152,2,15.56
lightrag,q050,1.0,1.0,0.8571,0.8169,0.3709,12647,217,2,17.58
lightrag,q051,1.0,1.0,1.0,0.952,0.6457,12482,156,2,15.49
lightrag,q052,1.0,0.8048,1.0,0.8438,0.2312,12566,260,2,18.85
lightrag,q058,1.0,1.0,0.9231,0.8057,0.3846,11721,271,2,20.22
lightrag,q059,1.0,1.0,0.375,0.693,0.5271,12111,179,2,15.65
lightrag,q068,1.0,0.6131,0.5294,0.8647,0.43,12402,306,2,19.85
lightrag,q069,1.0,1.0,0.8,0.7629,0.3607,12343,253,2,18.21
lightrag,q070,1.0,1.0,0.8333,0.8977,0.2793,12258,293,2,19.42
lightrag,q071,1.0,1.0,0.9583,0.8524,0.2866,12407,455,2,24.7
lightrag,q072,1.0,0.7654,1.0,0.9338,0.386,11816,384,2,21.79
lightrag,q073,1.0,1.0,1.0,0.8744,0.3386,12289,283,2,19.09
lightrag,q076,1.0,0.1461,0.8,0.6957,0.35,12126,125,2,14.03
lightrag,q077,1.0,0.6448,0.7143,0.8967,0.3275,11491,243,2,17.01
lightrag,q078,1.0,0.571,1.0,0.695,0.2959,11927,248,2,17.66
lightrag,q086,1.0,0.8829,1.0,1.0,0.3953,12003,223,2,16.87
lightrag,q087,1.0,1.0,1.0,0.9596,0.386,11826,236,2,17.07
lightrag,q088,1.0,0.8829,1.0,0.9808,0.3721,11955,206,2,16.13
lightrag,q090,0.0,0.0,0.0,0.8677,0.3736,12361,309,2,19.63
lightrag,q091,1.0,1.0,1.0,0.9994,0.377,12776,270,2,19.02
lightrag,q108,0.0,0.0,0.6,0.9734,0.2078,12318,138,2,16.26
lightrag,q109,0.0,0.0,1.0,0.7607,0.1619,12081,354,2,20.94
lightrag,q110,1.0,1.0,0.8889,0.8973,0.3425,11949,198,2,15.97
lightrag,q114,1.0,1.0,1.0,0.9459,0.5522,12674,177,2,16.01
lightrag,q115,1.0,1.0,1.0,0.7194,0.4324,12119,224,2,16.9'''
local = pd.read_csv(io.StringIO(LOCAL_CSV))
local.groupby("system")[["hit", "ndcg", "faithfulness", "answer_relevancy", "token_f1",
                         "prompt_tokens", "completion_tokens", "llm_calls", "latency_s"]].mean().round(3)